# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their `@id` fields as per the FAIR^2 standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure the latest version of `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
We load the dataset metadata and explore its contents using `mlcroissant`. The metadata provides key dataset-level information and the available record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset via its schema
dataset = mlc.Dataset(croissant_url)

# Metadata as a Python object
metadata = dataset.metadata  # Don't treat as dict! Use attribute access.

print(f"\033[1m{metadata.name}\033[0m\n{metadata.description}")
print(f"\nIdentifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Let's inspect the available record sets (`cr:RecordSet`), their associated `@id` fields, fields and columns. All references are shown by their `@id` for clarity and reproducibility.

In [ ]:
# List all available record sets with their @id
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in this dataset. Please check the schema or documentation.")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | Name: {rs.get('name', rs['@id'])}")

    # For demonstration, take the first record set (most datasets only have one main data table)
    record_set_id = record_sets[0]['@id']
    print(f"\nFields for record set @id: {record_set_id}")
    # Show field @id and names for this record set
    field_list = dataset.fields(record_set=record_set_id)
    for field in field_list:
        print(f"  - @id: {field['@id']} | Name: {field.get('name', field['@id'])} | Data type: {field.get('dataType', 'N/A')}")
    # Save field IDs to a list for later use
    field_ids = [field['@id'] for field in field_list]

## 3. Data Extraction
We now load data from a specific record set. We'll use the first record set found above and reference it by its `@id`, as well as print the DataFrame columns to show the available fields (referenced by their `@id`).

In [ ]:
# We'll extract all records from the main record set, using its @id
dataframes = {}
if record_sets:
    # You can change 'record_set_id' to refer to any other available record set @id
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nRecords from record set @id: {record_set_id}")
    print("Columns (@id of fields):", df.columns.tolist())
    df.head()
else:
    print('No data available to extract.')

## 4. Exploratory Data Analysis (EDA)
Let's perform some basic EDA steps: filtering on a numeric field, normalization, and grouping by a categorical field—**all references are by field `@id`**.

If you need to find which fields are numeric and suitable for analysis, check the printout of the previous fields. We'll pick an integer or float type field for demonstration.

In [ ]:
# For demonstration, let's pick the first numeric (float or int) field
import numpy as np
numeric_field_id = None
group_field_id = None

# Look for integer or float fields in the field_list
for field in dataset.fields(record_set=record_set_id):
    dt = field.get('dataType', '').lower()
    if ('int' in dt or 'float' in dt or 'number' in dt or 'duration' in dt) and numeric_field_id is None:
        numeric_field_id = field['@id']
    if ('sex' in field.get('name', '').lower() or 'gender' in field.get('name', '').lower()) and group_field_id is None:
        group_field_id = field['@id']

# Fallback for categorical group field
if group_field_id is None and field_ids:
    group_field_id = field_ids[-1]  # pick the last field

if numeric_field_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    # If the numeric field is actually stored as object, try to convert to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records for field '@id': {numeric_field_id} > {threshold:.2f}")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized values for field '@id': {numeric_field_id}")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by the grouping field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of '{numeric_field_id}' by field '@id': {group_field_id}")
        print(grouped_df)
else:
    print(f"No appropriate numeric field found in record set '@id': {record_set_id} for EDA.")

## 5. Visualization
Visualize the distribution of the chosen numeric variable and its relationship to the grouping field (e.g., Sex), using their `@id`s.

_If matplotlib is not installed, uncomment the pip line below._

In [ ]:
# !pip install matplotlib seaborn
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id in dataframes and numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[record_set_id][numeric_field_id], kde=True)
    plt.title(f"Distribution of field '@id': {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in dataframes[record_set_id].columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(data=dataframes[record_set_id], x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print('Cannot plot: data or numeric field not available.')

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze a FAIR dataset using the `mlcroissant` library while strictly referencing all data elements by their Croissant `@id`.

**Key steps included:**
- Loading the Croissant schema and metadata.
- Discovering available record sets and fields (all by `@id`).
- Loading tabular data from a selected record set.
- Performing normalization and group-wise aggregation on a numeric field.
- Visualizing field distributions and relationships.

_This workflow is fully reproducible and aligns with FAIR data principles, making it adaptable to new Croissant datasets._